In [14]:
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import torchvision
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torchsummary import summary
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np
import time

In [6]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size ,patch_size ,num_hiddens):
        super().__init__()
        self.num_patches= (img_size// patch_size ) ** 2
        self.conv = nn.Conv2d(3,num_hiddens,kernel_size=patch_size,stride=patch_size)

    def forward(self,x):
        return self.conv(x).flatten(2).transpose(1,2)


class ViTMLP(nn.Module):
    def __init__(self,mlp_num_hidden, mlp_num_outputs,dropout=0.5):
        super().__init__()
        self.dense1=nn.LazyLinear(mlp_num_hidden)
        self.gelu = nn.GELU()
        self.dropout1=nn.Dropout(dropout)
        self.dense2=nn.LazyLinear(mlp_num_outputs)
        self.dropout2=nn.Dropout(dropout)

    def forward(self,x):
        return self.dropout2(self.dense2(self.dropout1(self.gelu(self.dense1(x)))))


class ViTBlock(nn.Module):
    def __init__(self,num_hidden,norm_shape,mlp_num_hiddens,num_heads,dropout,use_bias=False):
        super().__init__()
        self.ln1= nn.LayerNorm(norm_shape)
        self.attention = nn.MultiheadAttention(num_hidden,num_heads,dropout,use_bias,batch_first=True)
        self.ln2= nn.LayerNorm(norm_shape)
        self.mlp= ViTMLP(mlp_num_hiddens,num_hidden,dropout)

    def forward(self,x):
        x2= self.ln1(x)
        attention_output,_=self.attention(x2,x2,x2)
        x = x + attention_output
        x2= self.ln2(x)
        mlp_output= self.mlp(x2)
        x = x + mlp_output
        return x



In [29]:
class ViT(nn.Module):
    def __init__(self, eta, n_iter, random_state, batch_size, img_size, patch_size,
                 num_hidden, norm_shape, mlp_num_hiddens, num_heads, num_blks,
                 num_classes=100, dropout=0.2):
        super().__init__()
        self.eta = eta
        self.n_iter = n_iter
        self.batch_size = batch_size
        self.random_state = random_state

        self.patch_embedding = PatchEmbedding(img_size, patch_size, num_hidden)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hidden))
        num_steps = self.patch_embedding.num_patches + 1
        self.pos_embedding = nn.Parameter(torch.randn(1, num_steps, num_hidden))
        self.drop_out = nn.Dropout(dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"{i}", ViTBlock(num_hidden, norm_shape, mlp_num_hiddens,
                                                    num_heads, dropout, use_bias=False))
        self.head = nn.Sequential(nn.LayerNorm(num_hidden), nn.Linear(num_hidden, num_classes))

        self.device = torch.device(
            "cuda" if torch.cuda.is_available()
            else "cpu"
        )
        print(f"Using device: {self.device}")

        self.train_losses_     = []
        self.train_accuracies_ = []
        self.val_losses_       = []
        self.val_accuracies_   = []
        self.to(self.device)

    def forward(self, x):
        x = self.patch_embedding(x)
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), 1)
        x = self.drop_out(x + self.pos_embedding)
        for blk in self.blks:
            x = blk(x)
        x = self.head(x[:, 0])
        return x

    def iter_mini_batch(self,X,y):
        dataset=TensorDataset(X,y)
        dataloader= DataLoader(dataset,batch_size=self.batch_size,shuffle=True)
        return dataloader

    def fit(self, X,y,X_test,y_test):
        torch.manual_seed(self.random_state)
        start_time = time.time()
        optimizer = optim.SGD(self.parameters(), lr=self.eta, momentum=0.9, weight_decay=1e-4)
        criterion= nn.CrossEntropyLoss()
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.n_iter)
        for epoch in range(self.n_iter):
            epoch_start= time.time()
            self.train()
            train_loader = self.iter_mini_batch(X,y)
            running_loss=0.0
            correct=0
            total =0

            for X_batch,y_batch in train_loader:
                X_batch= X_batch.to(self.device)
                y_batch = y_batch.to(self.device)
                optimizer.zero_grad()
                y_hat= self(X_batch)
                loss= criterion(y_hat,y_batch)
                loss.backward()
                optimizer.step()

                running_loss+=loss.item()
                preds=y_hat.argmax(dim=1)
                correct+= (preds == y_batch).sum().item()
                total += y_batch.numel()
            epoch_time = time.time()-epoch_start
            epoch_loss = running_loss/(len(train_loader))
            epoch_acc= 100 * correct/total

            self.train_losses_.append(epoch_loss)
            self.train_accuracies_.append(epoch_acc)

            val_loss, val_acc = self._evaluate(X_test, y_test, criterion)
            self.val_losses_.append(val_loss)
            self.val_accuracies_.append(val_acc)
            current_lr = optimizer.param_groups[0]['lr']

            print(f'Epoch {epoch+1:>3}/{self.n_iter} | '
                  f'Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.2f}% | '
                  f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% |'
                  f'lr: {current_lr:.6f} |'
                  f'Time: {epoch_time:.1f}s')

        self.training_time_ = time.time() - start_time    #stop the clock
        mins = self.training_time_ // 60
        secs = self.training_time_ % 60
        print(f"\nTraining complete: {int(mins)}m {secs:.1f}s")
        return self

    def _evaluate(self, X_test, y_test, criterion):
        test_loader=self.iter_mini_batch(X_test,y_test)
        self.eval()
        running_loss = 0.0
        correct      = 0
        total        = 0
        with torch.no_grad():
            for xin, target in test_loader:
                xin = xin.to(self.device)
                target = target.to(self.device)

                outputs = self(xin)
                loss=criterion(outputs.view(-1, outputs.size(-1)), target.view(-1))

                running_loss+= loss.item()
                predicted = outputs.argmax(dim=-1)
                total += target.numel()
                correct += (predicted == target).sum().item()
        val_loss = running_loss / len(test_loader)
        val_acc  = 100.0 * correct / total
        return val_loss, val_acc

    def predict(self, X):
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32)
        self.eval()
        with torch.no_grad():
            X        = X.to(self.device)
            logits   = self(X)
            _, preds = torch.max(logits, dim=1)
        return preds.cpu().numpy()
    def accuracy(self, X, y):
        if isinstance(y, torch.Tensor):
            y = y.numpy()
        preds = self.predict(X)
        return 100.0 * np.mean(preds == y)
    def training_summary(self):
      mins = int(self.training_time_ // 60)
      secs = self.training_time_ % 60
      params = sum(p.numel() for p in self.parameters())
      print(f"Parameters:    {params:,}")
      print(f"Training time: {mins}m {secs:.1f}s")
      print(f"Best val acc:  {max(self.val_accuracies_):.2f}%")
      print(f"Final test acc: call model.accuracy(X_test, y_test)")




In [8]:
CIFAR100_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR100_STD  = (0.2673, 0.2564, 0.2762)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

train_set = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

print(f"Train size: {len(train_set)}, Test size: {len(test_set)}")
print(f"Image shape: {train_set[0][0].shape}")

100%|██████████| 169M/169M [40:24<00:00, 69.7kB/s]


Train size: 50000, Test size: 10000
Image shape: torch.Size([3, 32, 32])


In [22]:
!pip install torchinfo -q
from torchinfo import summary

def get_complexity_stats(model, input_size=(1, 3, 32, 32)):
    stats = summary(model, input_size=input_size, verbose=0)
    params = stats.total_params
    macs = stats.total_mult_adds
    flops = macs * 2
    return params, flops

In [10]:
vit_configs = [
    {'name': 'C1_baseline', 'patch_size': 4, 'num_hidden': 256, 'num_blks': 4, 'num_heads': 4},
    {'name': 'C2_patch8', 'patch_size': 8, 'num_hidden': 256, 'num_blks': 4,'num_heads': 4},
    {'name': 'C3_embed512','patch_size': 4, 'num_hidden': 512,'num_blks': 4, 'num_heads': 4},
    {'name': 'C4_blocks8', 'patch_size': 4, 'num_hidden': 256,'num_blks': 8,'num_heads': 4},
    {'name': 'C5_heads8', 'patch_size': 4, 'num_hidden': 256,'num_blks': 4, 'num_heads': 8},
    {'name': 'C6_max', 'patch_size': 8, 'num_hidden': 512, 'num_blks': 8,'num_heads': 8},
    {'name': 'C7_patch8_wide','patch_size': 8, 'num_hidden': 512, 'num_blks': 4, 'num_heads': 4},
    {'name': 'C8_deep_wide', 'patch_size': 4, 'num_hidden': 512, 'num_blks': 8, 'num_heads': 8},
]

In [11]:
def dataset_to_tensors(dataset):
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    X, y = next(iter(loader))
    return X, y

X_train, y_train = dataset_to_tensors(train_set)
X_test, y_test = dataset_to_tensors(test_set)

print(X_train.shape, y_train.shape)   # (50000, 3, 32, 32), (50000,)
print(X_test.shape, y_test.shape)     # (10000, 3, 32, 32), (10000,)

torch.Size([50000, 3, 32, 32]) torch.Size([50000])
torch.Size([10000, 3, 32, 32]) torch.Size([10000])


In [ ]:

import pickle

vit_results = []

for cfg in vit_configs:
    print(f"\n{'='*70}\nTraining {cfg['name']}: {cfg}\n{'='*70}")

    model = ViT(
        eta=0.001, n_iter=30, batch_size=64, random_state=42,
        img_size=32, patch_size=cfg['patch_size'],
        num_hidden=cfg['num_hidden'],
        norm_shape=cfg['num_hidden'],
        mlp_num_hiddens=cfg['num_hidden'] * 4,
        num_heads=cfg['num_heads'],
        num_blks=cfg['num_blks'],
        num_classes=100,
        dropout=0.2,
    )

    params, flops = get_complexity_stats(model)

    model.fit(X_train, y_train, X_test, y_test)   # adjust to match your fit()'s actual signature

    result = {
        'config': cfg['name'],
        **cfg,
        'params': params,
        'flops': flops,
        'final_train_acc': model.train_accuracies_[-1],
        'final_val_acc': model.val_accuracies_[-1],
        'best_val_acc': max(model.val_accuracies_),
        'training_time_s': model.training_time_,
        'train_losses': model.train_losses_,
        'val_losses': model.val_losses_,
        'val_accuracies': model.val_accuracies_,
    }
    vit_results.append(result)

    # checkpoint after every single config — never lose completed work to a later crash
    with open('vit_sweep_results.pkl', 'wb') as f:
        pickle.dump(vit_results, f)
    print(f"Checkpointed after {cfg['name']}")

import pandas as pd
vit_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('train_losses', 'val_losses', 'val_accuracies')} for r in vit_results])
vit_df


Training C1_baseline: {'name': 'C1_baseline', 'patch_size': 4, 'num_hidden': 256, 'num_blks': 4, 'num_heads': 4}
Using device: cuda
Epoch   1/30 | Loss: 4.4875 | Acc: 2.79% | Val Loss: 4.2911 | Val Acc: 4.20% |lr: 0.001000 |Time: 32.2s
Epoch   2/30 | Loss: 4.2789 | Acc: 4.49% | Val Loss: 4.2181 | Val Acc: 5.45% |lr: 0.001000 |Time: 32.8s
Epoch   3/30 | Loss: 4.1825 | Acc: 5.93% | Val Loss: 4.0960 | Val Acc: 7.72% |lr: 0.001000 |Time: 31.6s
Epoch   4/30 | Loss: 4.0502 | Acc: 7.83% | Val Loss: 3.9847 | Val Acc: 8.97% |lr: 0.001000 |Time: 31.7s
Epoch   5/30 | Loss: 3.9364 | Acc: 9.47% | Val Loss: 3.8592 | Val Acc: 11.18% |lr: 0.001000 |Time: 31.7s
Epoch   6/30 | Loss: 3.8271 | Acc: 11.24% | Val Loss: 3.7494 | Val Acc: 12.76% |lr: 0.001000 |Time: 31.7s
Epoch   7/30 | Loss: 3.7419 | Acc: 12.64% | Val Loss: 3.6853 | Val Acc: 14.24% |lr: 0.001000 |Time: 31.7s
Epoch   8/30 | Loss: 3.6681 | Acc: 14.03% | Val Loss: 3.6004 | Val Acc: 15.56% |lr: 0.001000 |Time: 31.7s
Epoch   9/30 | Loss: 3.5932 